数据并不总是机器学习算法训练所需要的、已经处理完毕的最终格式。

我们使用 **transforms（数据变换）** 对数据做加工处理，使其适配训练要求。

所有 TorchVision 的数据集都提供两个参数：

- transform：用来处理图像特征
- target_transform：用来处理标签

二者接收可调用对象，里面存放变换逻辑。torchvision.transforms模块提供了大量开箱即用的常用变换操作。

FashionMNIST 的原始图像是 PIL 图片格式，标签是整数。训练时，我们需要图像转为归一化后的张量，标签需要变成独热编码张量。为完成这些变换，我们使用 torchvision.transforms.v2 API，搭配 torch.nn.functional.one_hot 函数。

In [ ]:
import torch
import torch.nn.functional as F
from torchvision import datasets
from torchvision.transforms import v2

ds = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
    target_transform=v2.Lambda(
        lambda y: F.one_hot(torch.tensor(y), num_classes=10).float()
    # 标签独热编码
    ),
)

**补充** ：

**关于ToImage () 与 ToDtype ()**：torchvision.transforms.v2 API 使用两步流程替代了旧版的ToTensor变换：
v2.ToImage 将 PIL 图像 或者 NumPy ndarray，转换成torchvision.tv_tensors.Image专用图像张量；
搭配scale=True参数的v2.ToDtype，把数据转为float32类型，同时将像素值缩放至[0.,1.]区间

**关于Lambda Transforms**：Lambda 变换可以执行任意用户自定义的 lambda 匿名函数。
这里我们使用torch.nn.functional.one_hot，把整数标签转换成长度为 10 的独热编码张量（数据集一共 10 个类别），再转为 float 浮点类型，匹配网络需要的数据类型。